<a href="https://colab.research.google.com/github/shabnamsattar/Frank-Wolf-Algorithm-for-Recommender-Systems/blob/main/FW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
import os
import pandas as pd
import numpy as np
from scipy.sparse import csr_array
from scipy.sparse.linalg import svds
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, r2_score
import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline
from time import time
from tqdm import tqdm, trange

In [ ]:
#BETA = 1e4
#N_ITER = 50
#FW_GAMMAS = [*map(lambda x: 2 / (2 + x) * 0.2, range(N_ITER))]

In [2]:
N_ITER = 50
alpha_FW = [2 / (1 + k) for k in range(N_ITER)]

Load and Prepare Data

In [3]:
#Download data from kaggle
path = kagglehub.dataset_download("netflix-inc/netflix-prize-data")

print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/netflix-inc/netflix-prize-data/versions/2


In [4]:
# Load data

df1= pd.read_csv(os.path.join(path, "combined_data_1.txt"), header= None, names= ['User_Id', 'Rate'], usecols= [0,1])
#df2= pd.read_csv(os.path.join(path, 'combined_data_2.txt'), header= None, names= ['User_Id', 'Rate'], usecols= [0,1])
#df3= pd.read_csv(os.path.join(path, 'combined_data_3.txt'), header= None, names= ['User_Id', 'Rate'], usecols= [0,1])
#df4= pd.read_csv(os.path.join(path, 'combined_data_4.txt'), header= None, names= ['User_Id', 'Rate'], usecols= [0,1])

df1['Rate'] = df1['Rate'].astype(float)
#df2['Rate'] = df2['Rate'].astype(float)
#df3['Rate'] = df3['Rate'].astype(float)
#df4['Rate'] = df4['Rate'].astype(float)



In [5]:
#print("Dataset shape:","\n", df1.shape,"\n", df2.shape,"\n", df3.shape,"\n", df4.shape)
print("Dataset shape:","\n", df1.shape)

Dataset shape: 
 (24058263, 2)


In [ ]:
# Combine multiple datasets into a single DataFrame
#dfs = [df1, df2]
#df = pd.concat(dfs, ignore_index=True)

# Display dataset shape
#print('Full dataset shape:', df.shape)

# Display dataset examples
#print("\n",df.head(10))


In [6]:
df = df1

In [7]:
# Extract indices of rows where 'Rate' is NaN
nan_indices = df[df['Rate'].isna()].index.tolist()

# Initialize a new column 'movie_Id' with default values
df['movie_Id'] = 0

# Assign movie IDs based on nan_indices
#nan_indices = df[df['Rate'].isna()].index.tolist()  # Assuming 'nan_indices' is defined this way
movie_id = 1

for i in range(len(nan_indices)):
    start = nan_indices[i] + 1
    end = nan_indices[i + 1] if i + 1 < len(nan_indices) else len(df)
    df.loc[start:end, 'movie_Id'] = movie_id
    movie_id += 1

#Print the updated DataFrame
print(df.shape)
print(df.head(10))


(24058263, 3)
   User_Id  Rate  movie_Id
0       1:   NaN         0
1  1488844   3.0         1
2   822109   5.0         1
3   885013   4.0         1
4    30878   4.0         1
5   823519   3.0         1
6   893988   3.0         1
7   124105   4.0         1
8  1248029   3.0         1
9  1842128   4.0         1


In [ ]:
#from google.colab import drive

# Step 1: Mount Google Drive
#drive.mount('/content/drive')

# Step 2: Specify the path where the CSV file will be saved in Google Drive
#file_path = '/content/drive/My Drive/netflix_data.csv'

# Step 3: Save the DataFrame as a CSV file
#df.to_csv(file_path, index=False)

#print(f"DataFrame has been saved to {file_path}")


In [ ]:
# Load the saved CSV file from Google Drive
#df_loaded = pd.read_csv('/content/drive/My Drive/netflix_data.csv')

# Display the loaded DataFrame
#print(df_loaded.head())


In [8]:
df_new = df.dropna(subset=['Rate'])
print(df_new.shape)
print(df_new.head(10))

(24053764, 3)
    User_Id  Rate  movie_Id
1   1488844   3.0         1
2    822109   5.0         1
3    885013   4.0         1
4     30878   4.0         1
5    823519   3.0         1
6    893988   3.0         1
7    124105   4.0         1
8   1248029   3.0         1
9   1842128   4.0         1
10  2238063   3.0         1


In [9]:
user_rating = df_new.drop_duplicates(['User_Id','movie_Id'])
user_rating.head(10)

,User_Id,Rate,movie_Id
1,1488844,3.0,1
2,822109,5.0,1
3,885013,4.0,1
4,30878,4.0,1
5,823519,3.0,1
6,893988,3.0,1
7,124105,4.0,1
8,1248029,3.0,1
9,1842128,4.0,1
10,2238063,3.0,1


In [ ]:
# Map user IDs and movie IDs to continuous integer indices
#user_ids = pd.unique(user_rating['User_Id'])
#movie_ids = pd.unique(user_rating['movie_Id'])
#user_map = {user_id: i for i, user_id in enumerate(user_ids)}
#movie_map = {movie_id: i for i, movie_id in enumerate(movie_ids)}

# Create rows, cols, and data arrays for COO matrix
#rows = user_rating['User_Id'].map(user_map)
#cols = user_rating['movie_Id'].map(movie_map)
#data = user_rating['Rate']

#matrix_shape = (len(user_ids), len(movie_ids))


In [ ]:
# Step 1: Extract unique user IDs and movie IDs
#user_ids = pd.unique(user_rating['User_Id'])
#movie_ids = pd.unique(user_rating['movie_Id'])

# Step 2: Create maps to assign new indices
#user_map = {user_id: i for i, user_id in enumerate(user_ids)}
#movie_map = {movie_id: i for i, movie_id in enumerate(movie_ids)}

# Step 3: Create rows and cols arrays with mapped indices
#rows = np.array([user_map[user_id] for user_id in user_ids])
#cols = np.array([movie_map[movie_id] for movie_id in movie_ids])
#data = user_rating['Rate']
# Step 4: Check dimensions
#print("Rows (mapped user indices):", rows, "Dimension:", len(rows))
#print("Cols (mapped movie indices):", cols, "Dimension:", len(cols))


In [ ]:
from scipy.sparse import coo_matrix, triu


# Assuming the dataset is in a DataFrame called user_rating
def create_user_movie_sparse_matrix(user_rating):
    # Step 1: Extract unique and sorted User_Id and movie_Id
    unique_users = sorted(user_rating['User_Id'].unique())
    unique_movies = sorted(user_rating['movie_Id'].unique())

    # Step 2: Create mapping from original User_Id and movie_Id to indices
    user_to_index = {user: idx for idx, user in enumerate(unique_users)}
    movie_to_index = {movie: idx for idx, movie in enumerate(unique_movies)}

    # Step 3: Map User_Id, movie_Id, and Rates to indices efficiently
    user_indices = user_rating['User_Id'].map(user_to_index)
    movie_indices = user_rating['movie_Id'].map(movie_to_index)
    rates = user_rating['Rate'].values

    # Step 4: Use coo_matrix for faster matrix creation
    user_movie_matrix = coo_matrix((rates, (user_indices, movie_indices)),
                                   shape=(len(unique_users), len(unique_movies)),
                                   dtype=np.float32)

    return user_movie_matrix.tocsr(), unique_users, unique_movies

def split_sparse_matrix(matrix, test_size=0.2, random_state=42):
    """
    Splits a sparse matrix into train and test sets.
    Args:
        matrix (scipy sparse matrix): The full sparse matrix to split.
        test_size (float): Proportion of data to be used as test set.
        random_state (int): Random state for reproducibility.

    Returns:
        train (scipy sparse matrix): Training set sparse matrix.
        test (scipy sparse matrix): Test set sparse matrix.
    """
    # Step 1: Extract COO format data for manipulation
    coo = matrix.tocoo()
    rows, cols, data = coo.row, coo.col, coo.data

    # Step 2: Split the indices into train and test
    train_idx, test_idx = train_test_split(np.arange(len(data)), test_size=test_size, random_state=random_state)

    # Step 3: Create train and test matrices using the split indices
    train_data = (data[train_idx], (rows[train_idx], cols[train_idx]))
    test_data = (data[test_idx], (rows[test_idx], cols[test_idx]))

    train = coo_matrix(train_data, shape=matrix.shape).tocsr()
    test = coo_matrix(test_data, shape=matrix.shape).tocsr()

    return train, test

# Example usage:
# Load the dataset (replace the path with your dataset's path)
# user_rating = pd.read_csv("path_to_your_dataset.csv")

# Call the function to create the sparse matrix
# sparse_matrix, sorted_users, sorted_movies = create_user_movie_sparse_matrix(user_rating)

# Split the matrix into train and test
# train_matrix, test_matrix = split_sparse_matrix(sparse_matrix, test_size=0.2)

# To display shapes and verify
# print("Train Matrix Shape:", train_matrix.shape)
# print("Test Matrix Shape:", test_matrix.shape)
# print("Train Non-Zero Count:", train_matrix.nnz)
# print("Test Non-Zero Count:", test_matrix.nnz)


In [15]:
sparse_matrix, sorted_users, sorted_movies = create_user_movie_sparse_matrix(user_rating)

In [16]:
train_matrix, test_matrix = split_sparse_matrix(sparse_matrix, test_size=0.2)

In [ ]:
# To display matrix shape and verify
print("Sparse Matrix Shape:", sparse_matrix.shape)
print("Sorted User IDs:", sorted_users)
print("Sorted Movie IDs:", sorted_movies)

In [17]:
print("Train Matrix Shape:", train_matrix.shape)
print("Test Matrix Shape:", test_matrix.shape)
print("Train Non-Zero Count:", train_matrix.nnz)
print("Test Non-Zero Count:", test_matrix.nnz)

Train Matrix Shape: (470758, 4499)
Test Matrix Shape: (470758, 4499)
Train Non-Zero Count: 19243011
Test Non-Zero Count: 4810753


taaaa inja be nazaram okaye

In [ ]:
matrix_shape = (len(user_ids), len(movie_ids))

In [ ]:
def make_sparse_data(values, rows, cols, shape):

    sparse_data = csr_array((values, (rows, cols)), shape=shape)

    return sparse_data

In [ ]:
#row_train, row_test, col_train, col_test, \
    #rate_train, rate_test = train_test_split(rows, cols, data, random_state=23)

In [ ]:
def make_sparse_data(values, rows, cols, shape):

    sparse_data = csr_array((values, (rows, cols)), shape=shape)

    return sparse_data

In [ ]:
def LMO (y, delta=1):

    u, s, vt = svds(-y, k=1, which='LM', solver='arpack')

    return delta * np.outer(u, vt)

In [ ]:
#def move_to_next_iterate (x_k, x_k_hat, alpha):

      #x_k_plus_1 = x_k + alpha * (x_k_hat - x_k)

    #return x_k_plus_1

In [ ]:
def FW_iteration (x_k, train_true_sparse, train_preds_sparse, delta, alpha):

    train_diff = train_preds_sparse - train_true_sparse
    train_loss = (train_diff ** 2).sum()
    gradiant = 2 * train_diff
    x_k_hat = LMO (gradiant, delta)
    x_k_plus_1 = x_k + alpha * (x_k_hat - x_k)

    return x_k_plus_1, train_loss

In [ ]:
def FW(x_init, train_ratings, train_rows, train_cols,
       val_ratings, val_rows, val_cols, n_iter, gammas,
       beta=1):


    train_losses = []
    train_r2s = []
    val_losses = []
    val_r2s = []
    acc_time = []
    last_time = 0
    x_t = x_init.copy()


    train_true_sparse = make_sparse_data(train_ratings,
                                         train_rows,
                                         train_cols,

                                         )

    tqdm_range = trange(n_iter, desc='Bar desc', leave=True)
    for i in tqdm_range:

        t1 = time()
        gamma = gammas[i]
        train_preds = x_t[train_rows, train_cols]
        val_preds = x_t[val_rows, val_cols]
        train_preds_sparse = make_sparse_data(train_preds,
                                              train_rows,
                                              train_cols,
                                              m_shape)

        r2_train = _eval_train(train_ratings, train_preds)
        r2_val, val_loss = _eval_validation(val_ratings, val_preds)

        x_t, train_loss = _fw_iter(x_t,
                                   train_true_sparse,
                                   train_preds_sparse,
                                   beta, gamma)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_r2s.append(r2_train)
        val_r2s.append(r2_val)

        t2 = time()

        time_diff = (t2 - t1) * 1000
        acc_time.append(time_diff + last_time)
        last_time += time_diff

        tqdm_range.set_description("Epoch %i, Train Loss %f, Val Loss %f)" %
                                   (i, train_loss, val_loss))
        tqdm_range.refresh()

    return x_t, train_losses, val_losses, train_r2s, val_r2s, acc_time

In [ ]:
fw_preds, fw_train_losses, fw_val_losses, \
    fw_r2s_train, fw_r2s_val, fw_time = FW(x_init, ratings_train,
                                           row_train, col_train,
                                           ratings_test, row_test,
                                           col_test, n_iter=N_ITER,
                                           gammas=FW_GAMMAS, beta=BETA)

In [ ]:
def generate_normalized_vector(size):
    '''
    generates vector with L2-norm = 1
    '''
    vec = np.random.normal(size=size)
    vec = vec / np.linalg.norm(vec)
    return vec


def generate_extreme_point(n_row, n_col, beta=1):
    '''
    generates extreme point
    '''
    u = generate_normalized_vector(n_row)
    v = generate_normalized_vector(n_col)
    p = beta * np.outer(u, v)
    return p




def generate_initial_point(n_row, n_col, betta=1):
    '''
    generates random matrix with nuclear norm <= betta
    as a convex combination of two extreme points
    '''
    p_1 = generate_extreme_point(n_row, n_col, betta)
    p_2 = generate_extreme_point(n_row, n_col, betta)
    gamma = np.random.random()
    initial_point = gamma * p_1 + (1 - gamma) * p_2
    return initial_point

In [ ]:
def visualize_results(train_losses, val_losses, r2s_train,
                      r2s_val, times, algo_name='Frank Wolfe'):

    '''
    plots validation curves
    '''

    fig, axes = plt.subplots(2, 2, figsize=(10, 10))

    axes[0][0].plot(train_losses, label='train')
    axes[0][0].plot(val_losses, label='validation')
    axes[0][0].legend()
    axes[0][0].grid()
    axes[0][0].set_xlabel('Iteration')
    axes[0][0].set_ylabel('Loss value')
    axes[0][0].set_title('%s. Loss vs iterations' % algo_name)

    axes[0][1].plot(r2s_train, label='train')
    axes[0][1].plot(r2s_val, label='validation')
    axes[0][1].legend()
    axes[0][1].grid()
    axes[0][1].set_xlabel('Iteration')
    axes[0][1].set_ylabel('R2')
    axes[0][1].set_title('%s. R2 vs iterations' % algo_name)

    axes[1][0].plot(times, train_losses, label='train')
    axes[1][0].plot(times, val_losses, label='validation')
    axes[1][0].legend()
    axes[1][0].grid()
    axes[1][0].set_xlabel('Time')
    axes[1][0].set_ylabel('Loss value')
    axes[1][0].set_title('%s. Loss vs time' % algo_name)

    axes[1][1].plot(times, r2s_train, label='train')
    axes[1][1].plot(times, r2s_val, label='validation')
    axes[1][1].legend()
    axes[1][1].grid()
    axes[1][1].set_xlabel('Time')
    axes[1][1].set_ylabel('R2')
    axes[1][1].set_title('%s. R2 vs time' % algo_name)

    plt.show()


def plot_confusion_matrix(true, preds, unique_ratings,
                          algo_name='Frank-Wolfe',
                          part='train'):
    '''
    plots confusion matrix
    '''
    # only integer targets are acceptable
    cm = confusion_matrix((true * 10).astype(int), (preds * 10).astype(int))
    df_cm = pd.DataFrame(cm, index=unique_ratings,
                         columns=unique_ratings)

    plt.figure(figsize=(10, 7))
    sns.heatmap(df_cm, annot=True, fmt='d')
    plt.title('%s. Confusion matrix for %s predictions' % (algo_name, part))
    plt.show()


def compare_results(fw_train_losses, pg_train_losses,
                    fw_time, pg_time):
    '''
    plots results of both algorithms
    '''
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    axes[0].plot(fw_train_losses, label='Frank-Wolfe')
    axes[0].plot(pg_train_losses, label='Projected gradient')
    axes[0].legend()
    axes[0].grid()
    axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel('Loss value')
    axes[0].set_title('Loss values vs iterations')

    axes[1].plot(fw_time, fw_train_losses, label='Frank Wolfe')
    axes[1].plot(pg_time, pg_train_losses, label='Projected gradient')
    axes[1].legend()
    axes[1].grid()
    axes[1].set_xlabel('Time')
    axes[1].set_ylabel('Loss value')
    axes[1].set_title('Loss values vs time')
    plt.show()

In [ ]:
def _eval_train(true, preds):
    '''
    evaluates train preds
    '''
    r2_train = r2_score(true, preds)
    return r2_train


def _eval_validation(true, preds):
    '''
    evaluates validation preds
    '''
    r2_val = r2_score(true, preds)
    val_loss = ((true - preds) ** 2).sum()
    return r2_val, val_loss

In [ ]:
x_init = generate_initial_point(m_shape[0], m_shape[1], BETA)

In [ ]:

fw_preds, fw_train_losses, fw_val_losses, \
    fw_r2s_train, fw_r2s_val, fw_time = FW(x_init, ratings_train,
                                           row_train, col_train,
                                           ratings_test, row_test,
                                           col_test, n_iter=N_ITER,
                                           gammas=FW_GAMMAS, beta=BETA)

In [ ]:
visualize_results(fw_train_losses, fw_val_losses, fw_r2s_train,
                  fw_r2s_val, fw_time, algo_name='Frank Wolfe')

In [ ]:
plot_confusion_matrix(ratings_train, train_prosessed_fw_preds, unique_ratings,
                      algo_name='Frank-Wolfe', part='train')

In [ ]:
prosessed_fw_preds = process_predictions(fw_preds)
train_prosessed_fw_preds = prosessed_fw_preds[row_train, col_train]
test_prosessed_fw_preds = prosessed_fw_preds[row_test, col_test]

In [ ]:
def process_predictions(preds):
    '''
    clips and rounds the preds
    '''
    processed_preds = np.round(preds * 2) / 2
    processed_preds[processed_preds < 0.5] = 0.5
    processed_preds[processed_preds > 5] = 5
    return processed_preds